### Extreme Gradient Boosting (XGBoost)

In [6]:
# Load Libraries
import warnings
warnings.filterwarnings("ignore")
import joblib
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from evaluation_script import evaluate_model

#### Load Data

In [7]:
# Load Processed Data
data = joblib.load('../building/dataset_bundle.pkl')

X_train = data['X_train']
y_train = data['y_train']
X_test = data['X_test']
y_test = data['y_test']


done


### Find the best parameters 

In [8]:
# Fin weight ratio
ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(ratio)


3.8105832832230906


In [10]:
# Declare parameters
param_grid = {
    'max_depth': [5, 7],
    'gamma': [0.01, 0.001],
    'min_child_weight': [1, 5],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [100, 200],
    'subsample': [0.8, 1],
    'colsample_bytree': [0.8, 1],
    'scale_pos_weight': [1, ratio]
}

In [11]:
# Create Search Space
xgb_grid = GridSearchCV(XGBClassifier(
                        objective = "binary:logistic",
                        eval_metric = "logloss",
                        use_label_encoder = False,
                        random_state = 42), 
                                param_grid, 
                                cv = 5, 
                                refit = True, 
                                scoring = 'f1',
                                verbose = 2)
print("done")

done


#### Fit model on training data

In [12]:
xgb_grid.fit(X_train , y_train)
best_model = xgb_grid.best_estimator_
best_model
print("data")

Fitting 5 folds for each of 256 candidates, totalling 1280 fits
[CV] END colsample_bytree=0.8, gamma=0.01, learning_rate=0.05, max_depth=5, min_child_weight=1, n_estimators=100, scale_pos_weight=1, subsample=0.8; total time=   0.7s
[CV] END colsample_bytree=0.8, gamma=0.01, learning_rate=0.05, max_depth=5, min_child_weight=1, n_estimators=100, scale_pos_weight=1, subsample=0.8; total time=   0.9s
[CV] END colsample_bytree=0.8, gamma=0.01, learning_rate=0.05, max_depth=5, min_child_weight=1, n_estimators=100, scale_pos_weight=1, subsample=0.8; total time=   0.6s
[CV] END colsample_bytree=0.8, gamma=0.01, learning_rate=0.05, max_depth=5, min_child_weight=1, n_estimators=100, scale_pos_weight=1, subsample=0.8; total time=   0.6s
[CV] END colsample_bytree=0.8, gamma=0.01, learning_rate=0.05, max_depth=5, min_child_weight=1, n_estimators=100, scale_pos_weight=1, subsample=0.8; total time=   0.6s
[CV] END colsample_bytree=0.8, gamma=0.01, learning_rate=0.05, max_depth=5, min_child_weight=1, 

#### Evaluate on testing 

In [13]:
evaluate_model(best_model, X_test, y_test)

{'Recall_Churn': 0.6898395721925134,
 'Precision_Churn': 0.48044692737430167,
 'F1_Churn': 0.566410537870472,
 'ROC_AUC': 0.8404667469134584}